# Setup and load all models from models/ folder

In [1]:
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"

model_names = ["RW", "DNS", "Ridge", "XGBoost"]

model_preds = {}
model_actuals = {}
model_metrics = {}

for name in model_names:
    path = MODEL_DIR / f"{name}.pkl"
    with open(path, "rb") as f:
        bundle = pickle.load(f)

    print(f"Loaded {name} from {path}, keys: {list(bundle.keys())}")

    model_preds[name]   = bundle["predictions"]
    model_actuals[name] = bundle["actuals"]
    model_metrics[name] = bundle["metrics"]   # <--- NEW

print("Models loaded:", list(model_preds.keys()))


Loaded RW from c:\Users\mango\Desktop\Bachelorarbeit\yield-curve-forecasting\models\RW.pkl, keys: ['predictions', 'actuals', 'metrics']
Loaded DNS from c:\Users\mango\Desktop\Bachelorarbeit\yield-curve-forecasting\models\DNS.pkl, keys: ['predictions', 'actuals', 'metrics']
Loaded Ridge from c:\Users\mango\Desktop\Bachelorarbeit\yield-curve-forecasting\models\Ridge.pkl, keys: ['predictions', 'actuals', 'metrics', 'hyperparameters']
Loaded XGBoost from c:\Users\mango\Desktop\Bachelorarbeit\yield-curve-forecasting\models\XGBoost.pkl, keys: ['predictions', 'actuals', 'metrics', 'hyperparameters']
Models loaded: ['RW', 'DNS', 'Ridge', 'XGBoost']


# Diebold-Marino Test

In [9]:
import numpy as np
from scipy.stats import norm

def dm_test(errors_model1, errors_model2, h=1, lag=None):
    """
    Diebold-Mariano test for equal predictive accuracy (squared-error loss).

    Parameters
    ----------
    errors_model1 : array-like
        Forecast errors of model 1 (y - y_hat1), length N.
    errors_model2 : array-like
        Forecast errors of model 2 (y - y_hat2), length N.
    h : int, optional
        Forecast horizon (in periods). Used only if lag is None.
    lag : int, optional
        Newey-West truncation lag for HAC variance.
        If None, lag is set to max(h-1, 0).

    Returns
    -------
    dm_stat : float
        Diebold-Mariano test statistic.
    p_value : float
        Two-sided p-value under asymptotic N(0,1).
    """

    e1 = np.asarray(errors_model1)
    e2 = np.asarray(errors_model2)

    # Drop NaNs in parallel
    mask = np.isfinite(e1) & np.isfinite(e2)
    e1 = e1[mask]
    e2 = e2[mask]

    if e1.shape != e2.shape:
        raise ValueError("Error series must have the same length after NaN removal.")

    N = len(e1)
    if N < 5:
        raise ValueError("Not enough observations for DM test.")

    # Squared-error loss
    L1 = e1 ** 2
    L2 = e2 ** 2

    # Loss differential
    d = L1 - L2
    d_bar = np.mean(d)

    # Newey–West HAC variance of d_t
    if lag is None:
        lag = max(h - 1, 0)

    d_centered = d - d_bar
    gamma0 = np.dot(d_centered, d_centered) / N
    var_d = gamma0

    for k in range(1, lag + 1):
        cov = np.dot(d_centered[k:], d_centered[:-k]) / N
        weight = 1.0 - k / (lag + 1)  # Bartlett weight
        var_d += 2.0 * weight * cov

    dm_stat = d_bar / np.sqrt(var_d / N)
    p_value = 2 * (1 - norm.cdf(np.abs(dm_stat)))

    return dm_stat, p_value


In [10]:
maturity_cols = ["DGS2", "DGS5", "DGS10"]
horizons = [1, 5, 10, 30]

dm_results = []

benchmark_name = "RW"

if benchmark_name not in model_actuals:
    raise ValueError(f"Benchmark model '{benchmark_name}' not loaded.")

# Use RW's actual series as canonical ground truth
rw_actual_series = model_actuals[benchmark_name]

for maturity in maturity_cols:
    for h in horizons:
        key = (maturity, h)

        # Use RW's actual series as canonical ground truth for this (maturity, h)
        if key not in rw_actual_series:
            continue

        actual = rw_actual_series[key].copy()
        actual.name = "actual"

        # Build a dict of aligned prediction series for all models that exist
        preds_for_key = {}
        for model_name, pred_dict in model_preds.items():
            if key not in pred_dict:
                continue
            preds_for_key[model_name] = pred_dict[key].rename(model_name)

        # Need at least RW + one other model with forecasts
        if benchmark_name not in preds_for_key or len(preds_for_key) < 2:
            continue

        # Compare each non-RW model against RW
        for model_name, pred_series in preds_for_key.items():
            if model_name == benchmark_name:
                continue

            # Pairwise alignment: actual, RW, and the other model
            df_pair = pd.concat(
                [
                    actual,
                    preds_for_key[benchmark_name],  # RW predictions
                    pred_series                     # other model predictions
                ],
                axis=1,
                join="inner"
            ).dropna()

            if df_pair.empty:
                continue

            # Forecast errors: e = actual - prediction
            err_rw    = df_pair["actual"] - df_pair[benchmark_name]
            err_other = df_pair["actual"] - df_pair[model_name]

            # Diebold–Mariano test (RW as Model 1, other model as Model 2)
            dm_stat, p_val = dm_test(err_rw.values, err_other.values, h=h)

            dm_results.append({
                "Maturity": maturity,
                "Horizon":  h,
                "Model_1":  benchmark_name,
                "Model_2":  model_name,
                "DM_stat":  dm_stat,
                "p_value":  p_val,
                "N":        len(df_pair)
            })

dm_results_df = pd.DataFrame(dm_results)
dm_results_df = dm_results_df.sort_values(["Maturity", "Horizon", "Model_2"])

display(dm_results_df)

,Maturity,Horizon,Model_1,Model_2,DM_stat,p_value,N
24,DGS10,1,RW,DNS,-12.676666,0.000000e+00,1724
25,DGS10,1,RW,Ridge,-12.725737,0.000000e+00,1724
26,DGS10,1,RW,XGBoost,-13.081711,0.000000e+00,1712
27,DGS10,5,RW,DNS,-8.635730,0.000000e+00,1716
28,DGS10,5,RW,Ridge,-8.619960,0.000000e+00,1716
29,DGS10,5,RW,XGBoost,-8.842232,0.000000e+00,1708
30,DGS10,10,RW,DNS,-6.674684,2.477663e-11,1706
31,DGS10,10,RW,Ridge,-6.684741,2.313327e-11,1706
32,DGS10,10,RW,XGBoost,-7.100229,1.245448e-12,1703
33,DGS10,30,RW,DNS,-4.283127,1.842846e-05,1666


# Root-Mean-Squared Error

In [11]:
# ----------------------------------------------------
# RMSE comparison table across models and horizons
# ----------------------------------------------------

rmse_tables = []

for model_name, df in model_metrics.items():
    if "RMSE" not in df.columns:
        print(f"⚠️ No RMSE column found for {model_name}, skipping.")
        continue

    rmse_tables.append(
        df[["RMSE"]].rename(columns={"RMSE": model_name})
    )

rmse_compare_df = pd.concat(rmse_tables, axis=1).sort_index()

display(rmse_compare_df)


RW       DNS     Ridge   XGBoost
Maturity Horizon                                        
DGS10    1        0.059004  0.059081  0.059137  0.061816
         5        0.126819  0.127669  0.127458  0.136784
         10       0.179402  0.181670  0.181782  0.193798
         30       0.328191  0.338187  0.346037  0.388307
DGS2     1        0.061250  0.061312  0.061421  0.065850
         5        0.131312  0.131994  0.132197  0.144119
         10       0.188868  0.190579  0.190964  0.210959
         30       0.359041  0.366263  0.369247  0.413400
DGS5     1        0.062598  0.062685  0.062806  0.065312
         5        0.134188  0.135193  0.135146  0.141190
         10       0.190236  0.192971  0.193134  0.206624
         30       0.349621  0.361938  0.364119  0.414612